# Task 3 : Constraint Satisfaction Problems with Propositional Logic

## Requirements
- Given a **m × n** matrix, each cell is a non-negative integer or blank.
- Each cell has **9 adjacent neighbours** (itself + 8 surrounding cells).
- Colour each cell **green** or **red** so the number of green neighbours matches the cell's clue.
- Blank cells have **no constraint**.
- Solve using **propositional logic** + **Glucose3** (PySAT).
- OOP model, compact and readable code.

## Install & Import

In [1]:
!pip install python-sat colorama -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 30.0 MB/s eta 0:00:00


In [2]:
from pysat.solvers import Glucose3
from itertools import combinations
from colorama import Fore, Back, Style, just_fix_windows_console
just_fix_windows_console()

## Architecture Overview
```
State            – full 2-D Cell grid (assigned colours)
Cell             – one grid cell: position, clue, colour
Problem          – loads input file; owns clause generation
SearchStrategy   – abstract base (ABC)
  └─ SATSolver   – Glucose3 implementation
Solution         – binds Problem + Strategy; solve / display / evaluate
```

## Cell Class

In [3]:
class Cell:
    """One cell: position, integer clue (or blank), and assigned colour."""

    BLANK = "."

    def __init__(self, row: int, col: int, clue):
        self.row      = row
        self.col      = col
        self.clue     = clue          # int or Cell.BLANK
        self.is_green = None          # True=green, False=red, None=unsolved

    def __str__(self) -> str:
        label = str(self.clue) if self.clue != self.BLANK else " "
        if self.is_green is None:
            return label
        colour = Back.GREEN if self.is_green else Back.RED
        return colour + Fore.WHITE + f" {label} " + Style.RESET_ALL

    def __repr__(self) -> str:
        return f"Cell(row={self.row}, col={self.col}, clue={self.clue!r}, is_green={self.is_green})"

## State Class

In [4]:
class State:
    """Wraps the full 2-D Cell grid and provides convenient iteration."""

    def __init__(self, cells: list):
        self.cells = cells
        self.rows  = len(cells)
        self.cols  = len(cells[0]) if cells else 0

    def __iter__(self):
        for row in self.cells:
            for cell in row:
                yield cell

    def __getitem__(self, idx):
        return self.cells[idx]

    def __repr__(self) -> str:
        return f"State(rows={self.rows}, cols={self.cols})"


## Problem Class

In [5]:
class Problem:
    """
    Environment: file I/O, grid geometry, CNF clause generation.

    Factory
    -------
    Problem.from_file(path)  – preferred constructor.
    """

    def __init__(self, state: State):
        self.state = state

    # ── factory ──────────────────────────────────────────────────────────────
    @classmethod
    def from_file(cls, filepath: str) -> "Problem":
        """Parse the text file and return a fully initialised Problem."""
        cells = []
        with open(filepath, "r") as fh:
            for r, line in enumerate(fh):
                row = []
                for c, token in enumerate(line.strip().split()):
                    clue = Cell.BLANK if token == "." else int(token)
                    row.append(Cell(r, c, clue))
                cells.append(row)
        return cls(State(cells))

    # ── properties ────────────────────────────────────────────────────────────
    @property
    def rows(self) -> int: return self.state.rows

    @property
    def cols(self) -> int: return self.state.cols

    # ── helpers ───────────────────────────────────────────────────────────────
    def cell_var(self, row: int, col: int) -> int:
        """Map (row, col) → positive SAT variable (1-indexed)."""
        return row * self.cols + col + 1

    def neighbors_of(self, row: int, col: int) -> list:
        """SAT variables of all 8-connected neighbours + self."""
        return [
            self.cell_var(r, c)
            for r in range(row - 1, row + 2)
            for c in range(col - 1, col + 2)
            if 0 <= r < self.rows and 0 <= c < self.cols
        ]

    # ── CNF clause generation ─────────────────────────────────────────────────
    def generate_clauses(self) -> list:
        """
        Encode constraints as CNF clauses.

        For a cell with clue k among N neighbours:
          • k == 0  → every neighbour must be red (N unit clauses).
          • k >= N  → every neighbour must be green (N unit clauses).
          • else    → exactly-k via at-least-k + at-most-k clauses.

        at-least-k : every (N−k+1)-subset must have ≥ 1 positive literal.
        at-most-k  : every  (k+1) -subset must have ≥ 1 negative literal.
        """
        clauses = []
        for cell in self.state:
            if cell.clue == Cell.BLANK:
                continue
            k    = cell.clue
            nbrs = self.neighbors_of(cell.row, cell.col)
            n    = len(nbrs)

            if k == 0:
                clauses.extend([[-v] for v in nbrs])
            elif k >= n:
                clauses.extend([[v]  for v in nbrs])
            else:
                for subset in combinations(nbrs, n - k + 1):   # at-least-k
                    clauses.append(list(subset))
                for subset in combinations(nbrs, k + 1):        # at-most-k
                    clauses.append([-v for v in subset])
        return clauses

## SearchStrategy (ABC) & SATSolver

In [6]:
from abc import ABC, abstractmethod

class SearchStrategy(ABC):
    """Abstract base – any solver must implement solve()."""

    @abstractmethod
    def solve(self, problem: Problem) -> bool:
        """Colour every cell in-place. Returns True if SAT, False if UNSAT."""


class SATSolver(SearchStrategy):
    """
    Concrete solver using Glucose3 (PySAT).

    Steps
    -----
    1. Ask Problem to generate CNF clauses.
    2. Feed clauses to Glucose3.
    3. Map the model back to cell.is_green values.
    """

    def solve(self, problem: Problem) -> bool:
        clauses = problem.generate_clauses()

        solver = Glucose3()
        for clause in clauses:
            solver.add_clause(clause)

        if not solver.solve():
            return False

        model_set = set(solver.get_model())
        for cell in problem.state:
            var           = problem.cell_var(cell.row, cell.col)
            cell.is_green = var in model_set
        return True

## Solution Class

In [11]:
class Solution:
    """
    Single entry point – binds Problem + Strategy.

    Usage
    -----
    problem  = Problem.from_file("input.txt")
    strategy = SATSolver()
    sol      = Solution(problem, strategy)
    sol.solve()
    sol.display()
    sol.evaluate()
    """

    def __init__(self, problem: Problem, strategy: SearchStrategy):
        self.problem  = problem
        self.strategy = strategy
        self._solved  = False

    def solve(self) -> bool:
        """Run the SAT solver. Returns True if a solution was found."""
        self._solved = self.strategy.solve(self.problem)
        return self._solved

    def display(self) -> None:
        """Pretty-print the coloured grid to the console."""
        if not self._solved:
            print("No solution yet – call solve() first.")
            return
        s = self.problem.state
        for r in range(s.rows):
            print(" " + "".join(str(s[r][c]) for c in range(s.cols)))
        print(Back.GREEN + "   " + Style.RESET_ALL + " Green  "
              + Back.RED  + "   " + Style.RESET_ALL + " Red\n")

    def evaluate(self) -> dict:
        """Verify every clued cell has exactly clue green neighbours."""
        if not self._solved:
            return {"error": "Not solved yet"}
        satisfied = violated = total = 0
        for cell in self.problem.state:
            if cell.clue == Cell.BLANK:
                continue
            total += 1
            green_count = sum(
                1 for v in self.problem.neighbors_of(cell.row, cell.col)
                if self.problem.state.cells[(v-1) // self.problem.cols]
                                           [(v-1) %  self.problem.cols].is_green
            )
            if green_count == cell.clue:
                satisfied += 1
            else:
                violated += 1
        result = {"satisfied": satisfied, "violated": violated,
                  "total_clued": total, "all_satisfied": violated == 0}
        print(f"Validation → {satisfied}/{total} constraints satisfied | all OK: {result['all_satisfied']}")
        return result

## Run

In [12]:
def main():
    # 1. Load problem from file
    problem  = Problem.from_file("input.txt")
    print(f"Grid loaded: {problem.rows} × {problem.cols}")

    # 2. Choose strategy and create solution
    strategy = SATSolver()
    sol      = Solution(problem, strategy)

    # 3. Solve
    print("Solving with Glucose3 …")
    if sol.solve():
        print("✓ Solution found!\n")
        sol.display()
        sol.evaluate()
    else:
        print("✗ No solution exists.")

main()

Grid loaded: 10 × 10
Solving with Glucose3 …
✓ Solution found!

     2  3        0             
              3     2        6 
        5     5  3     5  7  4 
     4     5     5     6     3 
        4     5     6        3 
           2     5             
  4     1           1  1       
  4     1           1     4    
              6              4 
     4  4              4       
    Green      Red

Validation → 36/36 constraints satisfied | all OK: True
